In [1]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

In [2]:
# -------------------------------------------
# Load dataset
# -------------------------------------------
# Load database
import pandas as pd
from pathlib import Path

# 2. Get Real Data Directory
dir = Path('/app/notebooks/data/synthetic/notebook_test/')
print(f"Using data from: {dir.name}")

# 3. Load problems data
df_episodes = pd.read_csv(dir / 'icare_episodes_anon.csv', parse_dates=['ADMISSION_DATE'])

print(df_episodes)

Using data from: notebook_test
    SUBJECT  SPELL_IDENTIFIER  ENCNTR_ID  EPISODE_IDENTIFIER ADMISSION_DATE  \
0     10001           2184571    6701228             7506958     2021-12-29   
1     10002           6736565    8976841             5323846     2023-12-31   
2     10003           4078312    4994004             1444028     2021-05-19   
3     10004           1733044    6276196             4584679     2020-03-03   
4     10005           7116961    8680007             6000356     2020-01-27   
..      ...               ...        ...                 ...            ...   
95    10096           8775561    5288105             4003193     2023-10-29   
96    10097           5926106    4066020             9765608     2022-11-29   
97    10098           1456895    7564854             5837989     2020-06-16   
98    10099           6991774    5578364             1856659     2022-04-20   
99    10100           7720887    3872849             3081424     2022-11-25   

   ADMISSION_TIME  A

In [3]:
# ---------------------------------------------------
# Derive phenotypes
# ---------------------------------------------------
# Libraries
import duckdb

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.registry.registry import DEFAULT_REGISTRY
from icare_risk.clinphen.registry.registry import register_from_yaml
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.engine.runner import FeatureMatrixBuilder

# Define paths
schema_cfg = '/app/src/icare_risk/config/icare/schema.yaml'
phenotype_cfg = '/app/src/icare_risk/config/icare/phenotypes.yaml'

# Load schema definition
schema = load_schema_from_yaml(schema_cfg)

# Create connection source
source = DuckDBSource(connection=duckdb.connect())

# Load the phenotypes definitions
DEFAULT_REGISTRY._specs.clear() # Clear before running
register_from_yaml(phenotype_cfg, DEFAULT_REGISTRY)

# Select phenotypes to compute
charlson = DEFAULT_REGISTRY.select_by_prefix('charlson_', return_names=True)
pitt = DEFAULT_REGISTRY.select_by_prefix('pitt_', return_names=True)
sirs = DEFAULT_REGISTRY.select_by_prefix('sirs_', return_names=True)
has = DEFAULT_REGISTRY.select_by_prefix('has_', return_names=True)
manual = [
    'has_congestive_heart_failure'
]

# Create matrix
builder = FeatureMatrixBuilder(
    schema=schema, source=source,
    registry=DEFAULT_REGISTRY,
    phenotypes=charlson
)

feature_matrix = builder.build(
    episodes=df_episodes,
    deduplicate_by=['SPELL_IDENTIFIER'],
    raise_on_error=True
)
display(feature_matrix.head(10))


Skipping 'definitions': missing module or function definition.
Skipping 'kim_hx_prior_antibiotics_90d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'tumbarello_hx_recent_abx_90d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'jones_hx_prior_antibiotics_30d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'gavaghan_age_ge_65': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_age_threshold'
Skipping 'gavaghan_hx_prior_antibiotics_fq_ceph_90d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'increment_bsi_not_urinary_flag': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_bsi_not_urinary'
Skipping 'increment_is_non_ecoli_flag': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_is_non_ecoli'
Skipping 'i

,subject,SPELL_IDENTIFIER,EPISODE_IDENTIFIER,admission_date,admission_time,AGE_AT_ADMISSION,discharge_date,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,...,charlson_hx_hemiplegia,charlson_hx_renal_mod_sev,charlson_hx_diabetes_comp,charlson_hx_cancer_solid,charlson_hx_leukemia,charlson_hx_lymphoma,charlson_hx_liver_mod_sev,charlson_hx_cancer_met,charlson_hx_aids,charlson_hx_hiv
encntr,,,,,,,,,,,,,,,,,,,,,
6701228,10001,2184571,7506958,2021-12-29,17:49:00,31,2022-01-22,140,oral surgery,none,...,0,0,0,0,0,0,0,0,0,0
8976841,10002,6736565,5323846,2023-12-31,13:08:00,20,2024-01-25,301,gastroenterology,none,...,0,0,0,0,0,0,0,0,0,0
4994004,10003,4078312,1444028,2021-05-19,02:27:00,35,2021-05-25,300,general medicine,unknown,...,0,0,0,0,0,0,0,0,0,0
6276196,10004,1733044,4584679,2020-03-03,16:11:00,74,2020-03-31,140,oral surgery,unknown,...,0,0,0,0,0,0,0,0,0,0
8680007,10005,7116961,6000356,2020-01-27,04:08:00,68,2020-02-16,320,cardiology,7,...,0,0,0,0,0,0,0,0,0,0
2457909,10006,2090119,9148088,2022-07-02,03:28:00,37,2022-07-10,140,oral surgery,7,...,0,0,0,0,0,0,0,0,0,0
4721780,10007,7320191,6689539,2022-11-05,21:56:00,76,2022-11-10,320,cardiology,5,...,0,0,0,0,0,0,0,0,0,0
9302201,10008,2091996,7699777,2021-04-02,12:15:00,35,2021-04-05,320,cardiology,7,...,0,0,0,0,0,0,0,0,0,0
5581399,10009,2098061,7182035,2022-08-21,05:00:00,77,2022-09-04,300,general medicine,10,...,0,0,0,0,0,0,0,0,0,0


In [4]:
cols_to_drop = [
    "encntr",
    "admission_date",
    "admission_time",
    "discharge_date",
    "index_admission",
    "index_discharge"
]

cols_to_drop = [col for col in cols_to_drop
    if col in feature_matrix.reset_index().columns]

# Show number of patients for each event name.
patient_counts = feature_matrix.reset_index() \
    .drop(columns=cols_to_drop) \
    .groupby("subject").any().sum() \
    .reset_index(name='patient_count') \
    .rename(columns={"index": "event_name"}) \
    .sort_values('patient_count', ascending=False)
display(patient_counts)

,event_name,patient_count
0,SPELL_IDENTIFIER,100
1,EPISODE_IDENTIFIER,100
2,AGE_AT_ADMISSION,100
3,MAIN_SPECIALTY_CODE,100
4,MAIN_SPECIALTY_CODE_DESC,100
5,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,100
15,charlson_hx_diabetes_uncomp,18
6,charlson_hx_mi,0
8,charlson_hx_pvd,0
7,charlson_hx_chf,0


In [7]:
feature_matrix = builder.build(
    episodes=df_episodes.head(10),
    deduplicate_by=['SPELL_IDENTIFIER'],
    raise_on_error=True,
    show_progress=True
)
display(feature_matrix.head(10))

Processing Episodes: 100%|██████████| 10/10 [00:00<00:00, 99.96it/s]


,subject,SPELL_IDENTIFIER,EPISODE_IDENTIFIER,admission_date,admission_time,AGE_AT_ADMISSION,discharge_date,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,...,charlson_hx_hemiplegia,charlson_hx_renal_mod_sev,charlson_hx_diabetes_comp,charlson_hx_cancer_solid,charlson_hx_leukemia,charlson_hx_lymphoma,charlson_hx_liver_mod_sev,charlson_hx_cancer_met,charlson_hx_aids,charlson_hx_hiv
encntr,,,,,,,,,,,,,,,,,,,,,
6701228,10001,2184571,7506958,2021-12-29,17:49:00,31,2022-01-22,140,oral surgery,none,...,0,0,0,0,0,0,0,0,0,0
8976841,10002,6736565,5323846,2023-12-31,13:08:00,20,2024-01-25,301,gastroenterology,none,...,0,0,0,0,0,0,0,0,0,0
4994004,10003,4078312,1444028,2021-05-19,02:27:00,35,2021-05-25,300,general medicine,unknown,...,0,0,0,0,0,0,0,0,0,0
6276196,10004,1733044,4584679,2020-03-03,16:11:00,74,2020-03-31,140,oral surgery,unknown,...,0,0,0,0,0,0,0,0,0,0
8680007,10005,7116961,6000356,2020-01-27,04:08:00,68,2020-02-16,320,cardiology,7,...,0,0,0,0,0,0,0,0,0,0
2457909,10006,2090119,9148088,2022-07-02,03:28:00,37,2022-07-10,140,oral surgery,7,...,0,0,0,0,0,0,0,0,0,0
4721780,10007,7320191,6689539,2022-11-05,21:56:00,76,2022-11-10,320,cardiology,5,...,0,0,0,0,0,0,0,0,0,0
9302201,10008,2091996,7699777,2021-04-02,12:15:00,35,2021-04-05,320,cardiology,7,...,0,0,0,0,0,0,0,0,0,0
5581399,10009,2098061,7182035,2022-08-21,05:00:00,77,2022-09-04,300,general medicine,10,...,0,0,0,0,0,0,0,0,0,0


In [8]:
cols_to_drop = [
    "encntr",
    "admission_date",
    "admission_time",
    "discharge_date",
    "index_admission",
    "index_discharge"
]

cols_to_drop = [col for col in cols_to_drop
    if col in feature_matrix.reset_index().columns]

# Show number of patients for each event name.
patient_counts = feature_matrix.reset_index() \
    .drop(columns=cols_to_drop) \
    .groupby("subject").any().sum() \
    .reset_index(name='patient_count') \
    .rename(columns={"index": "event_name"}) \
    .sort_values('patient_count', ascending=False)
display(patient_counts)

,event_name,patient_count
0,SPELL_IDENTIFIER,10
1,EPISODE_IDENTIFIER,10
2,AGE_AT_ADMISSION,10
3,MAIN_SPECIALTY_CODE,10
4,MAIN_SPECIALTY_CODE_DESC,10
5,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,10
15,charlson_hx_diabetes_uncomp,3
6,charlson_hx_mi,0
8,charlson_hx_pvd,0
7,charlson_hx_chf,0
